In [0]:
import requests
import pandas as pd
import os

# 1. Setup the Path inside your Repo
# os.getcwd() gets the current folder where your notebook lives in the Repo
current_repo_path = os.getcwd()
file_name = "sales_data.csv"
full_path = os.path.join(current_repo_path, file_name)

# 2. Download the data
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
response = requests.get(url)

if response.status_code == 200:
    # 3. Save it directly to your Repo folder
    with open(full_path, "wb") as f:
        f.write(response.content)
    print(f"File saved to Repo: {full_path}")
    
    # 4. BRIDGE: Read with Pandas first (since it's a local file)
    # Then convert to a Spark DataFrame so 'display' works
    temp_pdf = pd.read_csv(full_path)
    df = spark.createDataFrame(temp_pdf)
    
    # 5. Display the data
    display(df)
else:
    print(f"Failed to download. Status code: {response.status_code}")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS retail_db")
df.write.format("delta").mode("overwrite").saveAsTable("retail_db.bronze_sales")

In [0]:
# Import PySpark functions we need for transformations
from pyspark.sql.functions import col, current_timestamp

# 1. Read the data from our Bronze table into a PySpark DataFrame
bronze_df = spark.read.table("retail_db.bronze_sales")

# 2. Transform the data (Clean and Standardize)
silver_df = (
    bronze_df
    # Filter out bad data: keep only rows where total_bill is greater than 0
    .filter(col("total_bill") > 0)
    
    # Add a metadata column showing exactly when we processed this data
    .withColumn("processed_timestamp", current_timestamp())
)

# 3. Write this clean data into our Silver Layer as a new Delta table
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail_db.silver_sales")

# 4. Display the clean data to prove it worked!
display(spark.sql("SELECT * FROM retail_db.silver_sales LIMIT 5"))